In [9]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnableBranch

In [10]:
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.0)

In [12]:
math_prompt = ChatPromptTemplate.from_template(
    "You are a math professor. Solve this step-by-step: {question}"
)
coding_prompt = ChatPromptTemplate.from_template(
    "You are a senior developer. Write clean code for this: {question}"
)
general_prompt = ChatPromptTemplate.from_template(
    "You are a helpful assistant. Answer briefly: {question}"
)

classification_prompt = ChatPromptTemplate.from_template(
    "Classify the intent of the following questions with exactly one word : "
    "'math', 'coding', or 'generel'. \nQuestion: {question}"
)

In [13]:
classifier_chain = classification_prompt | llm | StrOutputParser()

In [14]:
# 3. Use RunnableBranch to act as an "If/Else" router
# Logic: (Condition function, Branch to execute)

branch = RunnableBranch(
    (lambda x: "math" in x["topic"].lower(), math_prompt | llm | StrOutputParser()),
    (lambda x: "coding" in x["topic"].lower(), coding_prompt | llm | StrOutputParser()),
    general_prompt | llm | StrOutputParser()
)

In [15]:
# 4. Assemble the Final Pipeline
# We pass the question to the classifier, assign the result to "topic", then route it.

def route_question(question: str):
    # Step A: Classify
    topic = classifier_chain.invoke({"question":question})
    print(f"--- Routed to: {topic.strip().upper()} ---")

    # Step B: Route
    return branch.invoke({"topic": topic, "question": question})

In [16]:
print(route_question("What is the derivative of x^2?"))
print("\n" + "="*40 + "\n")
print(route_question("Write a Python function to reverse a string."))

--- Routed to: MATH ---
To find the derivative of x^2, we'll use the power rule of differentiation. The power rule states that if we have a function of the form:

f(x) = x^n

Then the derivative of f(x) is:

f'(x) = n * x^(n-1)

In this case, we have:

f(x) = x^2

Here, n = 2. So, applying the power rule, we get:

f'(x) = 2 * x^(2-1)
f'(x) = 2 * x^1
f'(x) = 2x

Therefore, the derivative of x^2 is 2x.


--- Routed to: CODING ---
**Reversing a String in Python**

Here's a clean and efficient Python function to reverse a string:

```python
def reverse_string(input_str: str) -> str:
    """
    Reverses the input string.

    Args:
        input_str (str): The string to be reversed.

    Returns:
        str: The reversed string.
    """
    return input_str[::-1]
```

**Explanation**
---------------

This function uses Python's slice notation to reverse the string. The `[::-1]` slice means:

- `:`: Start from the beginning of the string.
- `:`: Go to the end of the string.
- `-1`: Step ba